# W07 tutorial — the Referee

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Referee:** one evaluation module the rest of the unit calls: give it a return series and it returns the headline table — annualised return, volatility, Sharpe, drawdown, turnover — gross and net.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of what this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
mkt = pd.read_parquet(DATA / "market_monthly.parquet")
ff  = pd.read_parquet(DATA / "ff_factors_monthly.parquet")
df = mkt.merge(ff, on="date", how="inner", validate="one_to_one").set_index("date")
print(df[["ewretd", "vwretd", "rf"]].describe().round(4))

Pause on the `describe()` output before moving on: `rf` has a mean of a couple of *tenths of a percent per month* (about 0.002 as a decimal). If your risk-free series ever has a mean like `4.85`, it is an annualised percentage and you are one subtraction away from the lecture's unit bug.

### 1. One function, six numbers

In [ ]:
AF = 12   # monthly data; use 252 for daily

def perf_table(r, rf, bench):
    """Annualised performance summary for one monthly return series."""
    ex     = r - rf
    active = r - bench
    W  = (1 + r).cumprod()
    dd = W / W.cummax() - 1
    return pd.Series({
        "Ann. return (arith.)": AF * r.mean(),
        "Ann. volatility":      np.sqrt(AF) * r.std(),
        "Sharpe (ann.)":        np.sqrt(AF) * ex.mean() / ex.std(),
        "Sortino (ann.)":       np.sqrt(AF) * ex.mean() / ex[ex < 0].std(),
        "Information (ann.)":   np.sqrt(AF) * active.mean() / active.std(),
        "Max drawdown":         dd.min(),
    })

report = pd.concat({
    "EW (1/N)":    perf_table(df["ewretd"], df["rf"], df["vwretd"]),
    "VW (market)": perf_table(df["vwretd"], df["rf"], df["vwretd"]),
}, axis=1)
print(report.round(3))

Both portfolios land in the 0.4–0.7 range for annualised Sharpe over 1990–2022, with maximum drawdowns of roughly −50% to −60%. The market's information ratio comes back as `NaN`, as it should: a benchmark has zero active return against itself, and 0/0 is not a number. A metrics function that *fails loudly* on a degenerate input is a feature.

Three conventions are worth saying out loud, because a table caption is not complete without them:

- annualised with √12 for volatilities and ratios, ×12 for means;
- Sortino uses the standard deviation of **negative excess returns** (several conventions exist — state yours);
- drawdown is computed from the **wealth index**, never from raw returns.

### 2. Sub-periods: the period-dependence check

In [ ]:
df["decade"] = (df.index.year // 10) * 10
g = (df["ewretd"] - df["rf"]).groupby(df["decade"])
print((np.sqrt(AF) * g.mean() / g.std()).round(2))

The 2000s come back at roughly half the Sharpe ratio of the 1990s and 2010s (about 0.35 against 0.75 and 0.70). Any full-sample number should travel with its sub-period breakdown — this is lie 4 (period dependence) made routine.

---

### 3. Pointing the Referee at a strategy

The Referee does not care where the returns come from — that is the whole point of writing it as a function of one series. Here we point it at the equal-weighted portfolio and score it against the client's value-weighted benchmark; assign any other monthly return series to `strategy` and the same table comes back. Two questions are worth settling before reading it: which single ratio is the *appropriate* headline for a client who holds the VW portfolio and mandates it as the benchmark, and which number in the table answers a client who asks "what is the worst that happens?"

In [ ]:
# The Referee is repointed by this one line: any monthly return Series indexed by date works
strategy = df["ewretd"]

print(perf_table(strategy, df["rf"], df["vwretd"]).round(3))

The appropriate headline is the **information ratio**. The client has a mandated benchmark, so the relevant question is active return per unit of tracking error against that benchmark. Sharpe would be the headline only if this strategy were the client's entire wealth; Treynor if it were a small sleeve of a broader diversified portfolio. Quoting the most flattering ratio instead of the appropriate one is exactly the behaviour the lecture warned about.

The "what is the worst that happens?" number is the **maximum drawdown** — the peak-to-trough loss the client would have lived through, with its dates. Volatility describes typical months; the drawdown describes the episode that makes clients fire managers.

---

### 4. Is it alpha?

A headline ratio says how good the returns look; it says nothing about where they came from. So the next thing the Referee needs is a regression of the strategy's excess returns on the CAPM and then the FF3 factors, with HAC (Newey–West) standard errors at `maxlags=6`, reporting annualised alphas next to their t-statistics. The question being answered is whether the strategy adds anything a factor ETF could not — this is the lecture's EW example, run end to end.

In [ ]:
import statsmodels.formula.api as smf

df["strat_ex"] = strategy - df["rf"]
capm = smf.ols("strat_ex ~ mkt_rf", data=df).fit(
    cov_type="HAC", cov_kwds={"maxlags": 6})
ff3 = smf.ols("strat_ex ~ mkt_rf + smb + hml", data=df).fit(
    cov_type="HAC", cov_kwds={"maxlags": 6})

for name, m in [("CAPM", capm), ("FF3 ", ff3)]:
    print(f"{name} alpha: {12 * m.params['Intercept']:+.2%} p.a. "
          f"(t = {m.tvalues['Intercept']:.2f})")
print(ff3.summary().tables[1])

The CAPM alpha comes out small and positive (about 1.3% a year, t ≈ 0.6) and FF3 barely moves it (about 1.0%, t ≈ 1.0), with an SMB loading of roughly 0.7–0.9 — what FF3 does is *name* the exposure, a size tilt wearing a disguise. The two cases read like this, here or on any other strategy:

- FF3 alpha insignificant (|t| < 2): the strategy's excess performance is a bundle of known factor exposures — the honest sentence is "the strategy earns its return through [size/value/market] tilts, available at ETF cost", not "the strategy generates alpha".
- FF3 alpha significant: a genuinely stronger claim — but Week 07's lies still apply (one sample, one period, and how many strategy variants were tried first?).

Both halves of the reporting rule bind here: alphas are quoted **annualised** (×12 — only ratios use √12) and **never without a t-statistic**.

---

### 5. The client's worst year

Ratios describe a typical month; clients live through episodes. So the Referee gets a second output: the wealth index plotted with its high-water mark, the drawdown underwater beneath it, and three numbers — the maximum drawdown, its trough date and, new this time, the **recovery date**, the first month in which wealth regains the high-water mark it lost. The distance from trough to recovery is how long the client spent underwater.

In [ ]:
import matplotlib.pyplot as plt

W   = (1 + strategy).cumprod()
hwm = W.cummax()
dd  = W / hwm - 1

fig, ax = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
ax[0].plot(W.index, W, color="#0F6BBA", label="Wealth")
ax[0].plot(hwm.index, hwm, color="#0F766E", ls="--", label="High-water mark")
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_ylabel("Growth of $1")
ax[1].fill_between(dd.index, dd, 0, color="#B91C1C", alpha=0.6)
ax[1].set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

trough = dd.idxmin()
peak_level = hwm.loc[trough]                       # the HWM to be regained
after = W.loc[trough:]
recovered = after[after >= peak_level]
print(f"Max drawdown: {dd.min():.1%}, trough {trough:%Y-%m}")
if len(recovered) > 0:
    rec = recovered.index[0]
    months = len(W.loc[trough:rec]) - 1            # .loc slice includes both ends, so subtract 1
    print(f"Recovered by {rec:%Y-%m} "
          f"({months} months after the trough)")
else:
    print("Still underwater at the end of the sample")

The drawdown bottoms at roughly −50% to −60% with the trough in early 2009, and recovery takes the better part of two years (22 months for the EW portfolio) — that duration, not the volatility, is what a client experiences. Note the logic: recovery means regaining the *previous high-water mark*, so we compare wealth after the trough with `hwm` at the trough. (The month count is the number of rows from the trough to the recovery date: the label slice `W.loc[trough:rec]` includes both endpoints, so we subtract one.)

Common error to avoid: running `cummax` on the raw return series. Drawdown is a property of cumulated wealth; on returns the number is meaningless.

---

### 6. Costs and the breakeven

Gross numbers are fiction, and the Referee is not finished until it charges for trading. Taking the lecture's figures — momentum tilt: 10.20% p.a. gross, turnover 5.0×; 1/N: 9.80% gross, turnover 0.20×; VW market: 9.10% gross, turnover 0.05× — and net return ≈ gross − turnover × cost, we rank the three net of the unit's 10 bps cost on two-way turnover (the W04 convention), then solve for the cost level at which momentum and 1/N tie. One question the table raises needs no code at all: why is the VW market's turnover so much lower than 1/N's, when both are "passive"?

In [ ]:
costs = pd.DataFrame({
    "strategy": ["Momentum tilt", "1/N (monthly rebal.)", "VW market"],
    "gross_pa": [0.1020, 0.0980, 0.0910],
    "turnover_pa": [5.00, 0.20, 0.05],
})
costs["net_pa"] = costs["gross_pa"] - costs["turnover_pa"] * 0.0010
print(costs.sort_values("net_pa", ascending=False).to_string(index=False))

breakeven = (0.1020 - 0.0980) / (5.00 - 0.20)
print(f"\nBreakeven two-way cost: {breakeven:.4%} ≈ {breakeven * 1e4:.1f} bps")

Net of 10 bps the ranking **flips**: 1/N (9.78%) beats momentum (9.70%), with the VW market at 9.10%. Gross, momentum led.

Setting $0.1020 - 5.00c = 0.0980 - 0.20c$ gives $c = 0.004/4.8 \approx 8.3$ **bps**. Below ~8 bps momentum wins; above, 1/N does. Our 10 bps assumption sits just past the flip — which is why performance is reported *net*, and never without the cost assumption it depends on.

As for the turnover asymmetry: value weights move *with* prices — when a stock rises, its market cap and its VW weight rise together, so the portfolio rebalances itself and only trades on entries, exits and share-count changes. Equal weights must be actively restored every month, selling winners and buying losers — turnover by design.

---

### 7. Verify the AI — finish the rigged-backtest hunt

The lecture's AI-generated backtest is reproduced below (with its two file paths pointed at `DATA`, so that it runs here). It runs without error, it prints a Sharpe ratio, and it is wrong three times over: two of the bugs would have *flattered* the result, and the third is so crude it betrays them. Two were identified in class — same-month `mktcap` weights (look-ahead), and `DTB3` in annualised percent subtracted from monthly decimal returns (units, and the crude one). **The third is still hidden.** Below we name it and say which of the six lies it is, evidence it with a computation rather than a code reading, rebuild the backtest correctly and check it against `vwretd` from `market_monthly.parquet`, and close with the ≤150-word audit note: evidence, fix, lesson.

In [ ]:
import pandas as pd

crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")
fred = pd.read_parquet(DATA / "fred_monthly.parquet")

# keep stocks with a full history so the panel is clean
last = crsp.groupby("permno")["date"].max()
survivors = last[last == crsp["date"].max()].index
crsp = crsp[crsp["permno"].isin(survivors)]

# value-weighted portfolio return
crsp["w"] = crsp["mktcap"] / crsp.groupby("date")["mktcap"].transform("sum")
port = (crsp["w"] * crsp["ret"]).groupby(crsp["date"]).sum()

# subtract the risk-free rate (3-month T-bill)
rf = fred.loc[fred["series_id"] == "DTB3"].set_index("date")["value"]
excess = port - rf.reindex(port.index)

sharpe = (12 ** 0.5) * excess.mean() / excess.std()
print(f"Sharpe ratio: {sharpe:.2f}")

**The third bug is the innocuous-looking "clean panel" filter — survivorship bias (lie 2).** `survivors` keeps only permnos whose *last* observation is the final month of the snapshot, i.e. stocks still alive in March 2022. Every bankruptcy, fraud delisting and acquisition between 1990 and 2022 is deleted from history, and Week 03 showed that performance-related delisting returns are large and negative. The comment "so the panel is clean" is the tell: AI assistants routinely dress a bias up as hygiene.

Reading the code is not evidence, so here is the survivor fingerprint — stock counts per year with and without the filter:

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet",
                       columns=["permno", "date", "ret", "mktcap"])
crsp["year"] = crsp["date"].dt.year

last = crsp.groupby("permno")["date"].max()
survivors = last[last == crsp["date"].max()].index

counts = pd.DataFrame({
    "all_stocks": crsp.groupby("year")["permno"].nunique(),
    "survivors_only": (crsp[crsp["permno"].isin(survivors)]
                       .groupby("year")["permno"].nunique()),
})
counts["share_kept"] = counts["survivors_only"] / counts["all_stocks"]
print(counts.loc[[1995, 2000, 2010, 2021]].to_string())

The survivor universe keeps only a small fraction of the stocks that actually traded in the 1990s — and the further back you look, the smaller the share — while the true universe *peaks* in the late 1990s. A universe that thins out going back in time, while the real market grew, is the fingerprint from the lecture.

Now the corrected backtest: lagged weights, no survivor filter, and a risk-free rate already in monthly decimals (the Ken French `rf`; converting `DTB3` with `/100/12` is the equivalent repair).

In [ ]:
crsp = crsp.sort_values(["permno", "date"])
# lagged market cap: this month's weight uses last month's size
crsp["mktcap_lag"] = crsp.groupby("permno")["mktcap"].shift(1)
ok = crsp.dropna(subset=["mktcap_lag", "ret"]).copy()
ok["w"] = ok["mktcap_lag"] / ok.groupby("date")["mktcap_lag"].transform("sum")
vw_fixed = (ok["w"] * ok["ret"]).groupby(ok["date"]).sum()   # monthly return, indexed by date

# df (section 1) already holds the Ken French rf and W04's vwretd, indexed by date,
# so Series arithmetic lines the months up for us
ex = vw_fixed - df["rf"]
print(f"Fixed annualised Sharpe: {np.sqrt(12) * ex.mean() / ex.std():.2f}")
print(f"Correlation with W04's vwretd: {vw_fixed.corr(df['vwretd']):.4f}")

The annualised Sharpe comes back into the market's honest 0.4–0.7 range and the correlation with `vwretd` is extremely close to 1 — the corrected code is Week 04's index construction rediscovered. (Tiny residual differences can come from how first-month observations are dropped.) The `shift(1)` lag assumes consecutive monthly observations per permno; Week 04 discussed handling gaps.

**The audit note itself, 89 words:** *"An AI-generated backtest of a value-weighted CRSP strategy reported an implausibly negative Sharpe ratio. I identified three defects: (1) portfolio weights used same-month market capitalisation, importing look-ahead bias; (2) the FRED DTB3 series, an annualised percentage, was subtracted from monthly decimal returns; (3) a 'clean panel' filter retained only stocks alive at the sample end, deleting all delistings — survivorship bias. Evidence: universe counts fall going back in time; rf.describe() shows a mean near 4.2. After lagging weights, converting units and removing the filter, results match the W04 index."*

---

### 8. Attribution for the client

The last thing the Referee owes a client is *where* the active return came from. Take a portfolio and its benchmark spanning three sectors: Financials — portfolio weight 50%, return 10%; benchmark weight 40%, return 8%. Consumer — 30% and 4%; benchmark 40% and 6%. Energy — 20% and 12%; benchmark 20% and 5%. The Brinson–Fachler decomposition splits the difference into allocation, selection and interaction, sector by sector and in total, and the three totals must sum to the active return.

In [ ]:
bf = pd.DataFrame({
    "sector":  ["Financials", "Consumer", "Energy"],
    "w_port":  [0.50, 0.30, 0.20], "r_port":  [0.10, 0.04, 0.12],
    "w_bench": [0.40, 0.40, 0.20], "r_bench": [0.08, 0.06, 0.05],
})
R = (bf["w_port"] * bf["r_port"]).sum()
B = (bf["w_bench"] * bf["r_bench"]).sum()

bf["allocation"]  = (bf["w_port"] - bf["w_bench"]) * (bf["r_bench"] - B)
bf["selection"]   = bf["w_bench"] * (bf["r_port"] - bf["r_bench"])
bf["interaction"] = (bf["w_port"] - bf["w_bench"]) * (bf["r_port"] - bf["r_bench"])
print(bf[["sector", "allocation", "selection", "interaction"]].round(4).to_string(index=False))
print(f"\nTotals: allocation {bf['allocation'].sum():+.2%}, "
      f"selection {bf['selection'].sum():+.2%}, "
      f"interaction {bf['interaction'].sum():+.2%} "
      f"→ active return {R - B:+.2%}")

Portfolio return 8.60%, benchmark 6.60%, active return **+2.00%**, decomposed as allocation **+0.20%**, selection **+1.40%**, interaction **+0.40%** — and the three totals sum to the active return exactly. They must: the decomposition is an algebraic identity (proof in Notes 07), so "do my effects sum?" is a free correctness check on any attribution table.

Reading the sectors: the outperformance is a *selection* story. Energy picks contribute +1.40% — the entire selection total — while Financials (+0.80%) and Consumer (−0.80%) cancel each other out; allocation decisions were nearly a wash.

The client sentence writes itself: "Outperformance of 2.0% came almost entirely from stock selection — notably in Energy — rather than from sector timing; our Consumer holdings underperformed their sector and are under review."

## Before you leave
#### ⚠️ Common mistakes this week
1. **Annualisation mixing** — ×12 on a Sharpe ratio instead of ×√12 (a factor-of-3.5 flattery), or a monthly mean paired with an annual risk-free rate. The single most common defect of the lot.
2. **Risk-free unit errors** — FRED's `DTB3` is an annualised percentage; monthly decimal returns need `value / 100 / 12` (or just use the Ken French `rf`). Run `rf.describe()` before trusting any merge.
3. **Drawdown from returns** — `cummax` on raw returns is meaningless; build the wealth index `(1 + r).cumprod()` first.
4. **Numerator/denominator mismatch** — excess returns on top with raw-return volatility below (or vice versa), and Sortino ratios with an unstated downside convention.
5. **Alphas without t-statistics** — or with plain OLS standard errors after HAC has been shown. An alpha without a t-statistic is an anecdote.